<a href="https://colab.research.google.com/github/ayushpaliwal1920/Deep_Learning/blob/main/23_HyperparameterTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [152]:
import pandas as pd
import numpy as np


In [153]:
df = pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [154]:
# checking correlation btwn data :

df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.466581
BloodPressure,0.065068
SkinThickness,0.074752
Insulin,0.130548
BMI,0.292695
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [155]:
# seperate data :

x = df.iloc[:,:-1].values
y = df.iloc[:,-1].values


In [156]:
# scaling :

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x = scaler.fit_transform(x)

In [157]:
# train test split

from sklearn.model_selection import train_test_split

x_train , x_test , y_train , y_test = train_test_split(x,y,test_size= 0.2, random_state=42)

In [158]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense , Dropout

In [159]:
model = Sequential()

model.add(Dense(32,activation='relu',input_dim=8))

model.add(Dense(1,activation='sigmoid'))



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [160]:
# compile

model.compile(optimizer="Adam" , loss = "binary_crossentropy" , metrics =['accuracy']);

In [161]:
model.fit(x_train,y_train , epochs = 50 , batch_size= 32,validation_data = (x_test , y_test))

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.4577 - loss: 0.7713 - val_accuracy: 0.5519 - val_loss: 0.6943
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5586 - loss: 0.6818 - val_accuracy: 0.6818 - val_loss: 0.6348
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6629 - loss: 0.6176 - val_accuracy: 0.7208 - val_loss: 0.5964
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7378 - loss: 0.5750 - val_accuracy: 0.7662 - val_loss: 0.5721
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7606 - loss: 0.5459 - val_accuracy: 0.7597 - val_loss: 0.5561
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7638 - loss: 0.5227 - val_accuracy: 0.7727 - val_loss: 0.5442
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7655 - loss: 0.5062 - val_accuracy: 0.7597 - val_loss: 0.5388
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4943 - val_accuracy: 0.7662 - val_los

# HyperParameter Tuning

In [162]:
# 1. how to select appropriate optimizer
# 2. how to select no. of nodes in a layer
# 3. howw to select no. of layers
# 4. All in one

In [163]:
pip install -U keras-tuner

In [164]:
import kerastuner as kt

# Finding Best optimizer :

In [165]:
def build_model(hp):
  model = Sequential()

  model.add(Dense(32,activation = 'relu' , input_dim = 8))

  model.add(Dense(1,activation = 'sigmoid'))

  # Tune the learning rate for the optimizer

  optimizer =  hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])

  model.compile(optimizer = optimizer  , loss = 'binary_crossentropy' , metrics = ['accuracy'])

  return model


In [166]:
# tuner object :

tuner = kt.RandomSearch(build_model,
                        objective='val_accuracy',
                        max_trials = 5)


Reloading Tuner from ./untitled_project/tuner0.json


In [167]:
tuner.search(x_train,y_train , epochs = 5, validation_data = (x_test,y_test))

In [168]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [169]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [170]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [171]:
model.fit(x_train,y_train,batch_size = 32 , epochs = 100 , initial_epoch =6 , validation_data = (x_test , y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7704 - loss: 0.5053 - val_accuracy: 0.7662 - val_loss: 0.5233
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4901 - val_accuracy: 0.7662 - val_loss: 0.5161
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7818 - loss: 0.4790 - val_accuracy: 0.7792 - val_loss: 0.5088
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7785 - loss: 0.4706 - val_accuracy: 0.7597 - val_loss: 0.5070
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7850 - loss: 0.4633 - val_accuracy: 0.7792 - val_loss: 0.5055
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4581 - val_accuracy: 0.7727 - val_loss: 0.5047
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4545 - val_accuracy: 0.7792 - val_loss: 0.5050
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4510 - val_accuracy: 0.78

# Finding best number of neurons in a single llayer :

In [172]:
def build_model_n(hp) :

  model = Sequential()

  units = hp.Int('units' ,min_value = 8, max_value = 128 , step = 8)

  model.add(Dense(units = units , activation = 'relu' , input_dim = 8))

  model.add(Dense(1,activation = 'sigmoid'))

  model.compile(optimizer = 'adam' , loss = 'binary_crossentropy' , metrics= ['accuracy'])

  return model

In [173]:
tuner_n = kt.RandomSearch(build_model_n,
                          objective= 'val_accuracy',
                          max_trials = 5,
                          directory = 'mydir',
                          project_name = 'Ayush')

Reloading Tuner from mydir/Ayush/tuner0.json


In [174]:
tuner_n.search(x_train,y_train , epochs = 5, validation_data = (x_test,y_test))

In [175]:
tuner_n.get_best_hyperparameters()[0].values

{'units': 56}

In [176]:
model = tuner.get_best_models(num_models= 1)[0]

In [177]:
model.fit(x_train,y_train,batch_size = 32 , epochs = 100 , initial_epoch=6, validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7687 - loss: 0.5053 - val_accuracy: 0.7597 - val_loss: 0.5223
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7801 - loss: 0.4905 - val_accuracy: 0.7792 - val_loss: 0.5164
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7801 - loss: 0.4800 - val_accuracy: 0.7662 - val_loss: 0.5113
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4707 - val_accuracy: 0.7727 - val_loss: 0.5070
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7915 - loss: 0.4642 - val_accuracy: 0.7792 - val_loss: 0.5033
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4589 - val_accuracy: 0.7727 - val_loss: 0.5056
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4549 - val_accuracy: 0.7792 - val_loss: 0.5060
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4513 - val_accuracy: 0.78

# Best number of layers :

In [178]:
def build_model_l(hp):

  model = Sequential()

  model.add(Dense(72,activation='relu' , input_dim = 8))

  for i in range(hp.Int('num_layers',min_value = 1 , max_value = 10)):

    model.add(Dense(72,activation = 'relu'))

  model.add(Dense(1,activation='sigmoid'))

  model.compile(optimizer='adam' , loss = 'binary_crossentropy',metrics=['accuracy'])

  return model


In [179]:
tuner_l = kt.RandomSearch(build_model_l,
                          objective = 'val_accuracy',
                          max_trials = 5,
                          directory = 'mydir',
                          project_name = 'num_of_layers')

Reloading Tuner from mydir/num_of_layers/tuner0.json


In [180]:
tuner_l.search(x_train,y_train,epochs = 5 ,validation_data=(x_test,y_test))

In [181]:
tuner_l.get_best_hyperparameters()[0].values

{'num_layers': 3}

In [182]:
model = tuner_l.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [183]:
model.fit(x_train,y_train,epochs=100,initial_epoch=6,validation_data=(x_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.7752 - loss: 0.4638 - val_accuracy: 0.7403 - val_loss: 0.5133
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7883 - loss: 0.4397 - val_accuracy: 0.7468 - val_loss: 0.5211
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7948 - loss: 0.4250 - val_accuracy: 0.7403 - val_loss: 0.5265
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8078 - loss: 0.4163 - val_accuracy: 0.7662 - val_loss: 0.5360
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7948 - loss: 0.4331 - val_accuracy: 0.7403 - val_loss: 0.5434
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8176 - loss: 0.3993 - val_accuracy: 0.7532 - val_loss: 0.5485
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8176 - loss: 0.4053 - val_accuracy: 0.7208 - val_loss: 0.5585
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8160 - loss: 0.3923 - val_accuracy: 0.72

# finding all best parameters :

In [195]:
def build_model_a(hp):

    model = Sequential()

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        if i == 0:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation_' + str(i), values=['relu','tanh','sigmoid']),
                input_dim=8
            ))
            model.add(Dropout(hp.Choice('dropout' +  str(i) , values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
        else:
            model.add(Dense(
                units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation_' + str(i), values=['relu','tanh','sigmoid'])
            ))
            model.add(Dropout(hp.Choice('dropout' +  str(i) , values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))


    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', values=['rmsprop','adam','sgd','nadam']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [196]:
tuner_a = kt.RandomSearch(build_model_a,
                          objective = 'val_accuracy',
                          max_trials = 3,
                          directory = 'mydir',
                          project_name = 'final')

Reloading Tuner from mydir/final/tuner0.json


In [197]:
tuner_a.search(x_train,y_train,epochs = 5 , validation_data=(x_test,y_test))

In [198]:
tuner_a.get_best_hyperparameters()[0].values

{'num_layers': 9,
 'units_0': 72,
 'activation_0': 'sigmoid',
 'optimizer': 'nadam',
 'units_1': 8,
 'activation_1': 'relu',
 'units_2': 8,
 'activation_2': 'relu',
 'units_3': 8,
 'activation_3': 'relu',
 'units_4': 8,
 'activation_4': 'relu',
 'units_5': 8,
 'activation_5': 'relu',
 'units_6': 8,
 'activation_6': 'relu',
 'units_7': 8,
 'activation_7': 'relu',
 'units_8': 8,
 'activation_8': 'relu'}

In [199]:
model = tuner_a.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 43 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [200]:
model.fit(x_train,y_train , epochs= 200 , initial_epoch= 6 , validation_data=(x_test,y_test) )

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - accuracy: 0.6270 - loss: 0.6527 - val_accuracy: 0.8052 - val_loss: 0.5384
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6531 - loss: 0.6269 - val_accuracy: 0.7987 - val_loss: 0.5640
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6612 - loss: 0.6196 - val_accuracy: 0.6429 - val_loss: 0.5663
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6580 - loss: 0.6173 - val_accuracy: 0.6429 - val_loss: 0.5764
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6629 - loss: 0.5855 - val_accuracy: 0.6429 - val_loss: 0.5678
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6596 - loss: 0.6080 - val_accuracy: 0.6429 - val_loss: 0.5669
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6694 - loss: 0.6081 - val_accuracy: 0.6429 - val_loss: 0.5721
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6629 - loss: 0.5886 - val_accurac